In [2]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

### Archivo DAR

In [80]:
df_DAR= pd.read_excel("C:/data/AUTOMATIZACION - ANULACIONES/DAR/DAR 2025-Julio a Octubre.xlsx", 
                      sheet_name=0, dtype={'CONTRATO': str, 'PRD': str})

In [81]:
df_DAR.drop(['CANAL','Mes','CLIENTE','Emisión'], axis=1, inplace=True)
df_DAR= df_DAR.rename(columns={'CONTRATO':'CERTIFICADO_BANCO', 'PRD':'CODIGO_PRODUCTO', 'FEC.ALTA':'FECHA_ALTA', 
                               'FEC.BAJA':'FECHA_BAJA', 'DIAS':'DIFERENCIA_DIAS','DIV':'MONEDA'})

In [82]:
df_DAR['FECHA_ALTA']= pd.to_datetime(df_DAR['FECHA_ALTA'],format='%Y-%m-%d', errors='coerce').dt.date
df_DAR['FECHA_BAJA']= pd.to_datetime(df_DAR['FECHA_BAJA'],format='%Y-%m-%d', errors='coerce').dt.date
df_DAR['DIFERENCIA_DIAS'] = (pd.to_datetime(df_DAR['FECHA_BAJA']) - pd.to_datetime(df_DAR['FECHA_ALTA'])).dt.days

In [83]:
df_DAR.head(3)

,CERTIFICADO_BANCO,CODIGO_PRODUCTO,PRODUCTO,FECHA_ALTA,FECHA_BAJA,DIFERENCIA_DIAS,MONEDA,PRIMA
0,00110118704000429349,800,SEGURO DE VIDA ...,2025-07-11,2025-07-15,4,USD,32.0
1,00110002324000089792,800,SEGURO DE VIDA ...,2025-07-15,2025-07-15,0,USD,12.0
2,00117794504006187952,803,SEG.CONT.PROTECCION ...,2025-07-02,2025-07-04,2,PEN,35.0


In [84]:
df_DAR.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7024 entries, 0 to 7023
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CERTIFICADO_BANCO  7024 non-null   object 
 1   CODIGO_PRODUCTO    7024 non-null   object 
 2   PRODUCTO           7024 non-null   object 
 3   FECHA_ALTA         7024 non-null   object 
 4   FECHA_BAJA         7024 non-null   object 
 5   DIFERENCIA_DIAS    7024 non-null   int64  
 6   MONEDA             7024 non-null   object 
 7   PRIMA              7024 non-null   float64
dtypes: float64(1), int64(1), object(6)
memory usage: 439.1+ KB


### Archivo FCR1

In [107]:
df_FCR1= pd.read_excel("C:/data/AUTOMATIZACION - ANULACIONES/FCR1/(Del 11.11.25 al 17.11.25) NACAR.xlsx", sheet_name=0)

In [108]:
df_FCR1.columns = (df_FCR1.columns.str.strip()  # quitar espacios al inicio/fin
                   .str.upper()  # opcional: todo en mayúsculas
                   .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)
df_FCR1 = df_FCR1.loc[:, ~df_FCR1.columns.duplicated()]

In [109]:
df_FCR1= df_FCR1.rename(columns={'N_MERO_DE_CONTRATO_DE_SEGURO':'CERTIFICADO_BANCO', 'TIPO_DE_SEGURO':'PRODUCTO', 'DIVISA':'MONEDA', 
                               'IMPORTE_ORIGINAL':'PRIMA'})

In [110]:
df_FCR1= df_FCR1[['CERTIFICADO_BANCO','PRODUCTO','MONEDA','PRIMA','GLOSA']]

In [111]:
df_FCR1.loc[df_FCR1['MONEDA'].str.strip().str.lower() == 'sol', 'MONEDA'] = 'PEN'
df_FCR1.loc[df_FCR1['MONEDA'].str.strip().str.lower() == 'dolar', 'MONEDA'] = 'USD'

In [112]:
df_FCR1['CERTIFICADO_BANCO'] = (df_FCR1['CERTIFICADO_BANCO'].str.strip().str.replace('-', '', regex=False))

In [113]:
df_FCR1.head(3)

,CERTIFICADO_BANCO,PRODUCTO,MONEDA,PRIMA,GLOSA
0,00117794524006461582,PROTECCIÓN MÚLTIPLE,PEN,35.0,ST-95577-PM-7794524006461582
1,00110368804000431564,MULTIRIESGO NEGOCIO,PEN,948.0,ST-95581-MN-0368804000431564
2,00117794564006471065,RENTA HOSPITALARIA,PEN,32.0,ST-95615-RH-7794564006471065


In [114]:
df_FCR1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CERTIFICADO_BANCO  143 non-null    object 
 1   PRODUCTO           143 non-null    object 
 2   MONEDA             143 non-null    object 
 3   PRIMA              143 non-null    float64
 4   GLOSA              143 non-null    object 
dtypes: float64(1), object(4)
memory usage: 5.7+ KB


### Archivo FCR2

In [129]:
df_FCR2= pd.read_excel("C:/data/AUTOMATIZACION - ANULACIONES/FCR2/0756_20.01.2026 al 22.01.2026.xlsx", sheet_name=0)
df_FCR2.columns = (df_FCR2.columns.str.strip()  # quitar espacios al inicio/fin
                   .str.upper()  # opcional: todo en mayúsculas
                   .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)

In [130]:
df_FCR2= df_FCR2.rename(columns={'NRO__CONTRATO':'CERTIFICADO_BANCO', 'DIVISA':'MONEDA', 
                                 'SALDO':'PRIMA', 'GLOSA_1':'GLOSA', 'NUMERO_DE_CONTROL':'NUMERO_CONTROL'})

In [131]:
df_FCR2= df_FCR2[['CERTIFICADO_BANCO','MONEDA','PRIMA','GLOSA','NUMERO_CONTROL']]
df_FCR2['CERTIFICADO_BANCO'] = (df_FCR2['CERTIFICADO_BANCO'].str.strip().str.replace('-', '', regex=False))
#df_FCR2 = df_FCR2.dropna(subset=['CERTIFICADO_BANCO'])

In [132]:
df_FCR2.head(3)

,CERTIFICADO_BANCO,MONEDA,PRIMA,GLOSA,NUMERO_CONTROL
0,00110266434001167793,PEN,300.05,DAPST-99372-SATA-0266434001167793,NaN
1,00110982694000755163,PEN,200.55,DAPST-99376-SATA-0982694000755163,NaN
2,00110222784000752056,PEN,262.73,DAPST-99378-SATA-0222784000752056,NaN


In [133]:
df_FCR2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CERTIFICADO_BANCO  89 non-null     object 
 1   MONEDA             91 non-null     object 
 2   PRIMA              91 non-null     float64
 3   GLOSA              91 non-null     object 
 4   NUMERO_CONTROL     2 non-null      float64
dtypes: float64(2), object(3)
memory usage: 3.7+ KB


### Archivo OMX

In [146]:
df_OMX= pd.read_excel("C:/data/AUTOMATIZACION - ANULACIONES/OMX/OMX Diciembre 2025.xlsx", sheet_name=0)
df_OMX.columns = (df_OMX.columns.str.strip()  # quitar espacios al inicio/fin
                   .str.upper()  # opcional: todo en mayúsculas
                   .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)

In [148]:
def renombrar_col_duplicadas(columnas):
    contador = {}
    nuevas_columnas = []
    for col in columnas:
        if col in contador:
            contador[col] += 1
            nuevas_columnas.append(f"{col}_{contador[col]}")
        else:
            contador[col] = 1
            nuevas_columnas.append(col)
    return nuevas_columnas

In [149]:
df_OMX.columns = renombrar_col_duplicadas(df_OMX.columns)

In [150]:
df_OMX= df_OMX.rename(columns={'CERTIFICADO_BCO_CON_D_GITOS_DE_CONTROL':'CERTIFICADO_BANCO', 
                               'MONTO':'PRIMA', 'DETALLE':'GLOSA', 'N__DE_OPERACION':'NUMERO_CONTROL',
                               '_RECLAMO':'NUMERO_RECLAMO','MONEDA_2':'MONEDA_OPERACION',
                               'MONTO_2':'PRIMA_OPERACION'})

In [151]:
df_OMX= df_OMX[['CERTIFICADO_BANCO','MONEDA','PRIMA','GLOSA','NUMERO_CONTROL', 
                'ORIGEN','CUENTA','FECHA_OPERACION','MONEDA_OPERACION',
                'PRIMA_OPERACION','FUENTE','NUMERO_RECLAMO']]

In [154]:
df_OMX.head(3)

,CERTIFICADO_BANCO,MONEDA,PRIMA,GLOSA,NUMERO_CONTROL,ORIGEN,CUENTA,FECHA_OPERACION,MONEDA_OPERACION,PRIMA_OPERACION,FUENTE,NUMERO_RECLAMO
0,00110178184000430476,USD,183.00,OMXP75693 DEV01781840004304,2177916,CONCILIACIÓN,BBVA 2265,2025-12-05,USD,61,LA/RI EMERGENTE - PAGO POR RECAUDO,NaN
1,00110229244001077725,USD,709.82,OMXP75693 DEV02292440010777,2178482,CONCILIACIÓN,BBVA 2265,2025-12-30,USD,709.82,LA/RI EMERGENTE - PAGO POR RECAUDO,NaN
2,00110281384001016037,USD,106.35,OMXP75693 DEV02813840010160,2178081,CONCILIACIÓN,BBVA 2265,2025-12-13,USD,106.35,LA/RI EMERGENTE - PAGO POR RECAUDO,NaN


In [155]:
df_OMX.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   CERTIFICADO_BANCO  235 non-null    object        
 1   MONEDA             252 non-null    object        
 2   PRIMA              252 non-null    float64       
 3   GLOSA              252 non-null    object        
 4   NUMERO_CONTROL     252 non-null    int64         
 5   ORIGEN             252 non-null    object        
 6   CUENTA             252 non-null    object        
 7   FECHA_OPERACION    252 non-null    datetime64[ns]
 8   MONEDA_OPERACION   238 non-null    object        
 9   PRIMA_OPERACION    238 non-null    object        
 10  FUENTE             252 non-null    object        
 11  NUMERO_RECLAMO     170 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(9)
memory usage: 23.8+ KB
